# Install and Import Libraries

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Load a small YOLOv8 model (optimized for speed)
model = YOLO('yolov8n.pt')

# Define Lane Polygons

In [ ]:
# Mark lane polygons interactively on img00001.jpg
ref_frame = cv2.imread("dataset/img00001.jpg")
if ref_frame is None:
    ref_frame = cv2.imread("img00001.jpg")
if ref_frame is None:
    raise FileNotFoundError("img00001.jpg not found in dataset/ or project root")

NUM_LANES = 3
lane_names = [f"Lane_{i + 1}" for i in range(NUM_LANES)]
lane_points = {name: [] for name in lane_names}
current_lane_idx = 0

window_name = "Lane Marker: left-click add | z undo | r reset | b back | n next lane | q finish"

def redraw_lane_canvas():
    canvas = ref_frame.copy()

    for i, name in enumerate(lane_names):
        pts = lane_points[name]
        if not pts:
            continue

        pts_np = np.array(pts, np.int32)
        is_active = i == current_lane_idx
        is_finished = i < current_lane_idx

        if len(pts) >= 2:
            if len(pts) >= 3:
                cv2.polylines(
                    canvas,
                    [pts_np],
                    isClosed=True,
                    color=(0, 255, 0) if (is_finished or not is_active) else (0, 255, 255),
                    thickness=3,
                )
            else:
                cv2.polylines(
                    canvas,
                    [pts_np],
                    isClosed=False,
                    color=(0, 200, 255),
                    thickness=2,
                )

        for p in pts:
            cv2.circle(canvas, p, 4, (0, 0, 255), -1)

        label_x = min(max(pts[0][0] + 8, 10), ref_frame.shape[1] - 150)
        label_y = min(max(pts[0][1] - 8, 20), ref_frame.shape[0] - 10)
        cv2.putText(
            canvas,
            name,
            (label_x, label_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

    if current_lane_idx < len(lane_names):
        active = lane_names[current_lane_idx]
        status = f"Marking: {active} ({len(lane_points[active])} points)"
    else:
        status = "All lanes marked. Press q to finish or b to review previous lane."

    cv2.putText(canvas, status, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(
        canvas,
        "Click to add points | z: undo | r: reset lane | b: back | n: next lane | q: finish",
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )
    return canvas


def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and current_lane_idx < len(lane_names):
        lane_points[lane_names[current_lane_idx]].append((x, y))


cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.setMouseCallback(window_name, on_mouse)
cv2.resizeWindow(window_name, ref_frame.shape[1], ref_frame.shape[0])
cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

while True:
    cv2.imshow(window_name, redraw_lane_canvas())
    key = cv2.waitKey(20) & 0xFF

    if key == ord("z") and current_lane_idx < len(lane_names):
        if lane_points[lane_names[current_lane_idx]]:
            lane_points[lane_names[current_lane_idx]].pop()

    elif key == ord("r") and current_lane_idx < len(lane_names):
        lane_points[lane_names[current_lane_idx]] = []

    elif key == ord("b") and current_lane_idx > 0:
        current_lane_idx -= 1

    elif key == ord("n") and current_lane_idx < len(lane_names):
        lane_name = lane_names[current_lane_idx]
        if len(lane_points[lane_name]) >= 3:
            current_lane_idx += 1
            if current_lane_idx >= len(lane_names):
                print("All lanes marked. Press q to finish.")
        else:
            print(f"{lane_name}: add at least 3 points before next lane")

    elif key == ord("q"):
        break
        # if all(len(lane_points[name]) >= 3 for name in lane_names):
        #     break
        # print("Please complete all lanes (>=3 points each)")

cv2.destroyAllWindows()

lane_polygons = {
    name: np.array(points, np.int32)
    for name, points in lane_points.items()
}

print("lane_polygons = {")
for name, poly in lane_polygons.items():
    print(f'    "{name}": np.array({poly.tolist()}, np.int32),')
print("}")

def draw_lanes(frame):
    for name, poly in lane_polygons.items():
        cv2.polylines(frame, [poly], isClosed=True, color=(0, 255, 0), thickness=2)
        cv2.putText(
            frame, name, tuple(poly[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2
        )
    return frame

# Showing Lane Polygons

In [ ]:
show_frame = cv2.imread("dataset/img00001.jpg")
if show_frame is None:
    show_frame = cv2.imread("img00001.jpg")
if show_frame is None:
    raise FileNotFoundError("img00001.jpg not found in dataset/ or project root")

if "lane_polygons" not in globals() or not lane_polygons:
    raise ValueError("lane_polygons is empty. Run the lane marking cell first.")

marked = show_frame.copy()
for lane_name, poly in lane_polygons.items():
    if len(poly) >= 2:
        cv2.polylines(marked, [poly], isClosed=True, color=(0, 255, 0), thickness=3)
    if len(poly) >= 1:
        x, y = poly[0]
        cv2.putText(
            marked, lane_name, (int(x), int(y) - 8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA
        )

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(marked, cv2.COLOR_BGR2RGB))
plt.title("Marked Lane Polylines")
plt.axis("off")
plt.show()

# Logic to Check if Vehicle is in Lane

In [ ]:
def get_lane_id(cx, cy):
    for name, poly in lane_polygons.items():
        # Check if the point (cx, cy) is inside the polygon
        is_inside = cv2.pointPolygonTest(poly, (cx, cy), False)
        if is_inside >= 0:
            return name
    return None

# Main Processing Loop

In [ ]:
# Load your sample image
image_path = "dataset/img00001.jpg"
frame = cv2.imread(image_path)
if frame is None:
    raise FileNotFoundError(f"{image_path} not found")

counts = {name: 0 for name in lane_polygons}

# Run YOLO detection
results = model(frame)

for result in results:
    for box in result.boxes:
        # Get coordinates and class
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = box.conf[0]
        cls = int(box.cls[0])

        # Only count cars/trucks/buses (COCO classes 2, 3, 5, 7)
        if cls in [2, 3, 5, 7] and conf > 0.4:
            # Calculate center point of the vehicle
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

            lane = get_lane_id(cx, cy)
            if lane:
                counts[lane] += 1
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

# Display results
frame = draw_lanes(frame)
print(f"Vehicle Counts: {counts}")
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

# Testing with Another Images

In [6]:
from pathlib import Path
import numpy as np
import cv2

# Change this if your images are in a different folder
dataset_dir = Path("dataset")

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
image_paths = sorted(
    [p for p in dataset_dir.rglob("*") if p.suffix.lower() in valid_exts]
)

if not image_paths:
    print(f"No images found in: {dataset_dir.resolve()}")
else:
    index = 0
    needs_redraw = True
    window_name = "Dataset Image Viewer (A/Left=Back, D/Right=Forward, Q/Esc=Quit)"

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    if hasattr(cv2, "WND_PROP_TOPMOST"):
        cv2.setWindowProperty(window_name, cv2.WND_PROP_TOPMOST, 1)

    while True:
        # Exit if user closed the window using the window close button
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        if needs_redraw:
            img_path = image_paths[index]
            img = cv2.imread(str(img_path))

            if img is None:
                canvas = 255 * np.ones((200, 900, 3), dtype=np.uint8)
                cv2.putText(canvas, f"Failed to load: {img_path}", (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                display_img = canvas
            else:
                display_img = img.copy()

            status_text = f"{index + 1}/{len(image_paths)}  |  {img_path}"
            cv2.putText(display_img, status_text, (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.imshow(window_name, display_img)
            needs_redraw = False

        key = cv2.waitKey(30) & 0xFF

        if key in [ord('d'), 83]:  # D or Right Arrow
            index = (index + 1) % len(image_paths)
            needs_redraw = True
        elif key in [ord('a'), 81]:  # A or Left Arrow
            index = (index - 1) % len(image_paths)
            needs_redraw = True
        elif key in [ord('q'), 27]:  # Q or Esc
            break

    cv2.destroyAllWindows()